In [1]:
"""
Colab & HF setup
- mount GDrive (store your access tokens there if ever)
- authenticate GitHub
- authenticate HuggingFace

If you're not on Colab, you can skip this.
If you're on Colab, run this once per runtime.
"""

!git clone https://github.com/Pacozabala/CSCI199.X-workbench
%cd CSCI199.X-workbench
!git fetch origin
!git checkout -b max/single-label-classifiers-approach --track origin/max/single-label-classifiers-approach

from google.colab import drive
drive.mount('/content/drive')

with open("/content/drive/MyDrive/keys/ghpat-colab.txt") as f:
  GH_TOKEN = f.read().strip()

import subprocess
REPO_URL = "https://github.com/Pacozabala/CSCI199.X-workbench"
GH_USER = "jeanmaxcacacho"

auth_url = REPO_URL.replace(
    "https://",
    f"https://{GH_USER}:{GH_TOKEN}@"
)
subprocess.run(["git", "remote", "set-url", "origin", auth_url], check=True)


with open("/content/drive/MyDrive/keys/hfpat-colab.txt") as f:
  HF_TOKEN = f.read().strip()

from huggingface_hub import login
login(token=HF_TOKEN)

Cloning into 'CSCI199.X-workbench'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 167 (delta 96), reused 115 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 5.60 MiB | 7.16 MiB/s, done.
Resolving deltas: 100% (96/96), done.
/content/CSCI199.X-workbench
Branch 'max/single-label-classifiers-approach' set up to track remote branch 'max/single-label-classifiers-approach' from 'origin'.
Switched to a new branch 'max/single-label-classifiers-approach'
Mounted at /content/drive



### Prepare Data

In [2]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/datasets/MFRC_polarity_pacoscript.csv")
df = df.dropna(subset=["polarity"])
df.head()

,text,subreddit,bucket,annotator,annotation,confidence,polarity
4,The Le Pen brand of conservatism and classical...,europe,French politics,annotator03,Authority,Somewhat Confident,authority.virtue
12,You are simplifying it. Islam is not the sole ...,europe,French politics,annotator03,Harm,Confident,harm.vice
13,You are simplifying it. Islam is not the sole ...,europe,French politics,annotator01,"Ingroup,Harm",Confident,"ingroup.vice,harm.vice"
14,You are simplifying it. Islam is not the sole ...,europe,French politics,annotator02,Harm,Confident,harm.vice
19,&gt; Valls is such a disgusting traitor to his...,europe,French politics,annotator04,"Ingroup,Purity,Authority",Confident,purity.vice


In [3]:
mfrc_ohe = df[["text", "polarity"]].copy()
mfrc_ohe.head()

,text,polarity
4,The Le Pen brand of conservatism and classical...,authority.virtue
12,You are simplifying it. Islam is not the sole ...,harm.vice
13,You are simplifying it. Islam is not the sole ...,"ingroup.vice,harm.vice"
14,You are simplifying it. Islam is not the sole ...,harm.vice
19,&gt; Valls is such a disgusting traitor to his...,purity.vice


In [4]:
labels = [
    "harm.virtue",
     "harm.vice",
     "authority.virtue",
     "authority.vice",
     "fairness.virtue",
     "fairness.vice",
     "loyalty.virtue",
     "loyalty.vice",
     "purity.virtue",
     "purity.vice"
     ]

for label in labels:
  mfrc_ohe[label] = 0


mfrc_ohe['polarity'] = mfrc_ohe['polarity'].str.replace('ingroup', 'loyalty', regex=False)
mfrc_ohe.head()

,text,polarity,harm.virtue,harm.vice,authority.virtue,authority.vice,fairness.virtue,fairness.vice,loyalty.virtue,loyalty.vice,purity.virtue,purity.vice
4,The Le Pen brand of conservatism and classical...,authority.virtue,0,0,0,0,0,0,0,0,0,0
12,You are simplifying it. Islam is not the sole ...,harm.vice,0,0,0,0,0,0,0,0,0,0
13,You are simplifying it. Islam is not the sole ...,"loyalty.vice,harm.vice",0,0,0,0,0,0,0,0,0,0
14,You are simplifying it. Islam is not the sole ...,harm.vice,0,0,0,0,0,0,0,0,0,0
19,&gt; Valls is such a disgusting traitor to his...,purity.vice,0,0,0,0,0,0,0,0,0,0


In [5]:
dummies = mfrc_ohe['polarity'].str.get_dummies(sep=',')
mfrc_ohe.update(dummies)

ground_truth = mfrc_ohe.copy()
ground_truth.head()

,text,polarity,harm.virtue,harm.vice,authority.virtue,authority.vice,fairness.virtue,fairness.vice,loyalty.virtue,loyalty.vice,purity.virtue,purity.vice
4,The Le Pen brand of conservatism and classical...,authority.virtue,0,0,1,0,0,0,0,0,0,0
12,You are simplifying it. Islam is not the sole ...,harm.vice,0,1,0,0,0,0,0,0,0,0
13,You are simplifying it. Islam is not the sole ...,"loyalty.vice,harm.vice",0,1,0,0,0,0,0,1,0,0
14,You are simplifying it. Islam is not the sole ...,harm.vice,0,1,0,0,0,0,0,0,0,0
19,&gt; Valls is such a disgusting traitor to his...,purity.vice,0,0,0,0,0,0,0,0,0,1


### Model Setup

MoralBERT is specified as such:
- ADAM optimizer is used with a learning rate of $5e^{-5}$
- there are $16$ batch sizes and $5$ epochs.

https://github.com/vjosapreniqi/MoralBERT/blob/main/MoralBert/Predict_mft_scores_from_the_MoralBERT_weights.ipynb

In [6]:
import numpy as np
import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin
from transformers import AutoModel, AutoTokenizer
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model = AutoModel.from_pretrained("bert-base-uncased").to(device)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
class MyModel(
    nn.Module,
    PyTorchModelHubMixin,
    # optionally, you can add metadata which gets pushed to the model card
    # repo_url="your-repo-url",
    pipeline_tag="text-classification",
    license="mit",
):
    def __init__(self, bert_model, moral_label=2):

        super(MyModel, self).__init__()
        self.bert = bert_model
        bert_dim = 768
        self.invariant_trans = nn.Linear(768, 768)
        self.moral_classification = nn.Sequential(nn.Linear(768,768),
                                                      nn.ReLU(),
                                                      nn.Linear(768, moral_label))

    def forward(self, input_ids, token_type_ids, attention_mask):
        pooled_output = self.bert(input_ids,
                                token_type_ids = token_type_ids,
                                attention_mask = attention_mask).last_hidden_state[:,0,:]


        pooled_output = self.invariant_trans(pooled_output)


        logits = self.moral_classification(pooled_output)

        return logits

In [8]:
def preprocessing(input_text, tokenizer):
    return tokenizer(
                        input_text,
                        add_special_tokens = True,
                        max_length = 150,
                        padding = 'max_length',
                        return_attention_mask = True,
                        return_token_type_ids = True,  # Add this line
                        return_tensors = 'pt',
                        truncation=True
                   )

In [9]:
# Example list of sentences
sentences = [
    "This is good, but I do not understand it!",
    "I care a lot for my health and well-being",
    "You have btrayed your country!!",
    "Breaking the law is bad."
    # Add more sentences as needed
]

# the list of Moral (MFT) values
mft_values = ["care", "harm", "fairness", "cheating", "loyalty", "betrayal",
              "authority", "subversion", "purity", "degradation"]


# function to load the model, predict the score, and return the second value
def get_model_score(sentence, mft):
    repo_name = f"vjosap/moralBERT-predict-{mft}-in-text"

    # loading the model
    model = MyModel.from_pretrained(repo_name, bert_model=bert_model).to(device)

    # preprocessing the text
    encodeds = preprocessing(sentence, tokenizer)
    encodeds = {k: v.to(device) for k, v in encodeds.items()}

    # predicting the mft score
    output = model(**encodeds)
    score = F.softmax(output, dim=1)

    # extracting and return the second value from the tensor
    mft_value = score[0, 1].item()

    return mft_value

# initialising a list to accumulate the results
results = []

# sequential execution of predictions
for sentence in sentences:
    # dictionary to store scores for the current sentence
    sentence_scores = {"sentence": sentence}

    # iterate through each MFT model and get the score
    for mft in mft_values:
        sentence_scores[mft] = get_model_score(sentence, mft)

    results.append(sentence_scores)

results_df = pd.DataFrame(results)

# display the final results
results_df

# save the DataFrame to a CSV file
# results_df.to_csv("moral_foundation_scores.csv", index=False)

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

,sentence,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation
0,"This is good, but I do not understand it!",0.017906,0.011827,0.000711,0.000608,0.001132,0.000901,0.000606,0.002593,0.001033,0.007981
1,I care a lot for my health and well-being,0.913170,0.000821,0.002522,0.000750,0.001102,0.010791,0.001195,0.001110,0.002080,0.008218
2,You have btrayed your country!!,0.091193,0.001212,0.003562,0.423067,0.992904,0.770573,0.002084,0.003547,0.000779,0.008177
3,Breaking the law is bad.,0.019675,0.083457,0.000525,0.002635,0.000824,0.003747,0.925532,0.990706,0.001234,0.025948


### Inference on `mfrc_ohe`, implement batch inference and what not.

In [10]:
models = {}
for mft in mft_values:
    repo_name = f"vjosap/moralBERT-predict-{mft}-in-text"
    models[mft] = MyModel.from_pretrained(repo_name, bert_model=bert_model).to(device)

def get_batch_scores(sentences, mft, batch_size=1024):
    model = models[mft]
    model.eval()
    all_scores = []

    for i in range(0, len(sentences), batch_size):
        batch = list(sentences[i:i+batch_size])
        encodeds = tokenizer(batch, add_special_tokens=True, max_length=150,
                             padding='max_length', truncation=True,
                             return_attention_mask=True, return_token_type_ids=True,
                             return_tensors='pt')
        encodeds = {k: v.to(device) for k, v in encodeds.items()}

        with torch.no_grad():
            output = models[mft](**encodeds)
            scores = F.softmax(output, dim=1)[:, 1].tolist()
        all_scores.extend(scores)

    return all_scores

import tqdm
input_texts = list(mfrc_ohe["text"])
results = {"text": list(input_texts)}
for mft in tqdm.tqdm(mft_values, desc="Processing MFT models"):
    results[mft] = get_batch_scores(input_texts, mft)

inference_results_df = pd.DataFrame(results)

Processing MFT models: 100%|██████████| 10/10 [01:26<00:00,  8.66s/it]


In [11]:
inference_results_df.head()
# inference_results_df.to_csv("/content/drive/MyDrive/MoralBERT-raws.csv", index=False)

,text,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation
0,The Le Pen brand of conservatism and classical...,0.377618,0.330923,0.430411,0.383453,0.664201,0.574285,0.349016,0.162718,0.452946,0.007505
1,You are simplifying it. Islam is not the sole ...,0.358845,0.410381,0.490878,0.443466,0.607938,0.526880,0.409065,0.186394,0.405181,0.008056
2,You are simplifying it. Islam is not the sole ...,0.358845,0.410381,0.490878,0.443466,0.607938,0.526880,0.409065,0.186394,0.405181,0.008056
3,You are simplifying it. Islam is not the sole ...,0.358845,0.410381,0.490878,0.443466,0.607938,0.526880,0.409065,0.186394,0.405181,0.008056
4,&gt; Valls is such a disgusting traitor to his...,0.370607,0.621017,0.395839,0.295754,0.247063,0.179034,0.450139,0.569077,0.578682,0.968955


In [12]:
inference_results_df[mft_values] = (inference_results_df[mft_values] >= 0.5).astype(int)
inference_results_df.head()
# inference_results_df.to_csv("/content/drive/MyDrive/MoralBERT-normalized.csv", index=False)

,text,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation
0,The Le Pen brand of conservatism and classical...,0,0,0,0,1,1,0,0,0,0
1,You are simplifying it. Islam is not the sole ...,0,0,0,0,1,1,0,0,0,0
2,You are simplifying it. Islam is not the sole ...,0,0,0,0,1,1,0,0,0,0
3,You are simplifying it. Islam is not the sole ...,0,0,0,0,1,1,0,0,0,0
4,&gt; Valls is such a disgusting traitor to his...,0,1,0,0,0,0,0,1,1,1
